# 09 · Точки интереса и сопоставление: исследовательские стенды
Практикум к лекции 10. Девять стендов: отклик → детекция → патч → канонизация → метрика → фильтр → полные конвейеры → покрытие → RANSAC. CPU, без обучения. Плановый маршрут с записями — 90–120 минут; можно разделить на две части. NASA photo загружает ~0.8 МБ с SHA-256; остальные входы строятся локально.

Перед запуском запишите прогноз. Измените один фактор, нажмите **«Вычислить»**, сравните число и рисунок и объясните причину. Начальное состояние вычисляется автоматически. Задания находятся непосредственно под каждым стендом. Выполните четыре пункта каждого стенда (36 записей, внутри пункта возможны несколько запусков).

Таблица записи: `стенд/пункт | прогноз | параметр до→после | измерения до→после | объяснение`. Число точек, близость дескрипторов и геометрическая правильность — разные результаты.

In [ ]:
import importlib.util, subprocess, sys
requirements={'numpy':'numpy>=1.26,<3','cv2':'opencv-python-headless==4.12.0.88','matplotlib':'matplotlib>=3.8,<4','ipywidgets':'ipywidgets>=8,<9','threadpoolctl':'threadpoolctl>=3.5,<4'}
missing=[package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import cv2
# В текущем Colab штатные варианты OpenCV конфликтуют: cv2 5 не содержит AKAZE.
# Исправляем только одноразовую среду Colab; локальную среду не меняем.
try:
    import google.colab as _colab_runtime
except ImportError:
    _colab_runtime=None
if _colab_runtime is not None and (cv2.__version__.split('.')[0]!='4' or not hasattr(cv2,'AKAZE_create')):
    import importlib.metadata as _opencv_metadata
    _variants=[]
    for _variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:_opencv_metadata.version(_variant);_variants.append(_variant)
        except _opencv_metadata.PackageNotFoundError:pass
    if _variants:subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*_variants])
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','opencv-python-headless==4.12.0.88'])
    raise RuntimeError('OpenCV 4.12 установлен. В Colab выберите «Среда выполнения → Перезапустить сеанс», затем выполните ноутбук сверху.')
if not hasattr(cv2,'AKAZE_create'):
    import importlib.metadata as meta
    installed=[]
    for variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try: meta.version(variant); installed.append(variant)
        except meta.PackageNotFoundError: pass
    if len(installed)==1:
        instruction=f'В отдельной ячейке выполните %pip install {installed[0]}==4.12.0.88 и перезапустите ядро. Сохраняйте только этот вариант OpenCV.'
    else:
        instruction='Найдено несколько вариантов OpenCV либо нестандартная сборка. Создайте чистую среду с одним вариантом opencv-python-headless==4.12.0.88; затем выберите ее kernel.'
    raise RuntimeError('Нужен OpenCV 4.x с AKAZE. '+instruction)
from threadpoolctl import threadpool_limits
_thread_limit=threadpool_limits(limits=1)
try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass
%matplotlib inline

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import time, json, csv, hashlib, tempfile, os, urllib.request, platform
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi': 105, 'font.size': 10})
assert all(hasattr(cv2, name) for name in ['SIFT_create','ORB_create','AKAZE_create'])
PHOTO_URL='https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/astronaut.png'
PHOTO_SHA='88431cd9653ccd539741b555fb0a46b61558b301d4110412b5bc28b5e3ea6cb5'

def load_photo():
    """NASA photograph, public domain; cached outside the notebook directory."""
    p=Path(os.environ.get('INTEREST_PHOTO_PATH', str(Path(tempfile.gettempdir())/'applied-ai-interest-points'/'astronaut.png')))
    if not p.exists():
        p.parent.mkdir(parents=True,exist_ok=True)
        with urllib.request.urlopen(PHOTO_URL,timeout=60) as response: data=response.read()
        if hashlib.sha256(data).hexdigest()!=PHOTO_SHA: raise ValueError('Download SHA-256 mismatch')
        p.write_bytes(data)
    if hashlib.sha256(p.read_bytes()).hexdigest()!=PHOTO_SHA: raise ValueError('Cached image SHA-256 mismatch')
    img=cv2.imdecode(np.frombuffer(p.read_bytes(),np.uint8),cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img,(384,384),interpolation=cv2.INTER_AREA)

def facade(seed=0,repeated=False,size=384):
    """Authored planar synthetic facade, not a photograph or a model prediction."""
    rng=np.random.default_rng(seed); im=np.full((size,size),190,np.uint8)
    for y in range(24,size-30,48):
        for x in range(24,size-30,48):
            cv2.rectangle(im,(x,y),(x+26,y+30),40,2)
            cv2.line(im,(x+13,y),(x+13,y+30),60,2)
            cv2.line(im,(x,y+15),(x+26,y+15),60,2)
            if not repeated:
                cv2.circle(im,(x+int(rng.integers(4,23)),y+int(rng.integers(4,27))),int(rng.integers(2,6)),int(rng.integers(0,150)),-1)
    if not repeated:
        noise=rng.normal(0,4,im.shape)
        im=np.uint8(np.clip(im+noise,0,255))
        cv2.putText(im,'A'+str(seed),(14,size-10),cv2.FONT_HERSHEY_SIMPLEX,.65,30,2)
    return im

def make_pair(im,angle=12.,scale=.92,dx=6.,dy=-3.,perspective=.00015,blur=.5,gain=1.):
    h,w=im.shape; A=np.vstack([cv2.getRotationMatrix2D(((w-1)/2,(h-1)/2),angle,scale),[0,0,1]])
    A[:2,2]+=[dx,dy]
    P=np.array([[1,0,0],[0,1,0],[perspective,-.6*perspective,1.]])
    H=P@A; H/=H[2,2]
    target=cv2.warpPerspective(im,H,(w,h),flags=cv2.INTER_LINEAR,borderValue=128)
    if blur>0: target=cv2.GaussianBlur(target,(0,0),blur)
    target=np.uint8(np.clip(target.astype(float)*gain,0,255))
    return im,target,H

def extract(im,method,budget=600):
    if method=='ORB': detector=cv2.ORB_create(nfeatures=budget,WTA_K=2);metric='hamming'
    elif method=='SIFT': detector=cv2.SIFT_create(nfeatures=budget);metric='l2'
    elif method=='AKAZE': detector=cv2.AKAZE_create();metric='hamming' # default MLDB
    else: raise ValueError(method)
    kp,desc=detector.detectAndCompute(im,None)
    order=sorted(range(len(kp)),key=lambda i:(-kp[i].response,kp[i].pt[0],kp[i].pt[1]))[:budget]
    pts=np.array([kp[i].pt for i in order],float).reshape(-1,2)
    if desc is None: desc=np.empty((0,detector.descriptorSize()),np.float32 if metric=='l2' else np.uint8)
    else: desc=desc[order]
    return pts,desc,metric

def draw_matches(a,b,p,q,pairs,correct=None,title='',limit=75,path=None):
    h,w=a.shape; canvas=np.hstack([a,b]);fig,ax=plt.subplots(figsize=(11,5),constrained_layout=True)
    ax.imshow(canvas,cmap='gray',vmin=0,vmax=255)
    for k,(i,j) in enumerate(pairs[:limit]):
        color='#10b981' if correct is None or correct[k] else '#f43f5e'
        ax.plot([p[i,0],q[j,0]+w],[p[i,1],q[j,1]],color=color,lw=.65,alpha=.8)
        ax.scatter([p[i,0],q[j,0]+w],[p[i,1],q[j,1]],c=color,s=5)
    ax.set_title(title+' | green: GT error ≤ 3 px; red: incorrect');ax.axis('off')
    if path: fig.savefig(path,dpi=140)
    plt.show();plt.close(fig)

def write_csv(path,rows):
    with Path(path).open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
print('OpenCV',cv2.__version__,'CPU threads',cv2.getNumThreads())

In [ ]:
import ipywidgets as W
from IPython.display import display
from IPython.utils.capture import capture_output
PANELS={}

def panel(name,fn,controls):
    out=W.Output();state={'calls':0,'last':None}
    def refresh(change=None):
        with capture_output(display=True) as captured:
            result=fn(**{key:control.value for key,control in controls.items()})
        items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata} for item in captured.outputs]
        if captured.stdout:items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
        out.outputs=tuple(items);state['last']=result;state['calls']+=1
    # Parameters are staged; one explicit run makes comparisons reproducible.
    for control in controls.values():
        if hasattr(control,'continuous_update'):control.continuous_update=False
        control.layout.width='285px';control.style.description_width='initial'
    button=W.Button(description='Вычислить',button_style='primary');button.on_click(refresh)
    PANELS[name]={'controls':controls,'state':state,'button':button,'refresh':refresh}
    display(W.VBox([W.HTML('Измените параметры и нажмите «Вычислить».'),W.HBox(list(controls.values()),layout=W.Layout(flex_flow='row wrap')),button,out]));refresh()

def view(images,titles):
    fig,axs=plt.subplots(1,len(images),figsize=(12,3.6),constrained_layout=True)
    for ax,im,title in zip(np.atleast_1d(axs),images,titles):
        ax.imshow(im,cmap='gray');ax.set_title(title);ax.axis('off')
    return fig,np.atleast_1d(axs)

def finish(fig,report):
    plt.show();plt.close(fig);print(json.dumps(report,ensure_ascii=False,allow_nan=False));return report

def project(p,H):
    p=np.asarray(p,float).reshape(-1,2)
    v=np.c_[p,np.ones(len(p))]@H.T
    return v[:,:2]/v[:,2:]

def coverage(p,side=384,cells=4):
    p=np.asarray(p).reshape(-1,2);p=p[(p>=0).all(1)&(p<side).all(1)]
    return len(np.unique(np.floor(p*cells/side).astype(int),axis=0))/cells**2

def image_source(source,seed=4):
    if source=='NASA photo':return load_photo()
    if source=='flat':return np.full((384,384),128,np.uint8)
    return facade(seed,repeated=source=='repeated')

def show_response(pattern,angle,contrast,noise,window,k,criterion,seed):
    y,x=np.mgrid[-40:41,-40:41];t=np.deg2rad(angle);u=x*np.cos(t)+y*np.sin(t);v=-x*np.sin(t)+y*np.cos(t)
    z=np.ones_like(u)*.5 if pattern=='flat' else (u>0).astype(float)
    if pattern=='corner':z=((u>0)&(v>0)).astype(float)
    rng=np.random.default_rng(seed);im=np.clip(.5+contrast*(z-.5)+rng.normal(0,noise,z.shape),0,1).astype('float32')
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,ksize=3)/8;gy=cv2.Sobel(im,cv2.CV_32F,0,1,ksize=3)/8
    r=window//2;s=np.s_[40-r:41+r,40-r:41+r];A=np.c_[gx[s].ravel(),gy[s].ravel()];M=A.T@A;ev=np.linalg.eigvalsh(M)
    response=cv2.cornerHarris(im,window,3,k) if criterion=='Harris' else cv2.cornerMinEigenVal(im,window,3)
    fig,axs=view([im,response],['Input patch','Response map (OpenCV normalized)']);axs[0].add_patch(plt.Rectangle((40-r,40-r),window,window,fill=False,color='red'))
    return finish(fig,{'eigenvalues (sum)':ev.tolist(),'Harris (sum)':float(np.linalg.det(M)-k*np.trace(M)**2),'Shi-Tomasi (sum)':float(ev[0]),'rank':int(np.linalg.matrix_rank(M,tol=1e-6))})

def show_detector(method,quality,distance,blur,pattern,angle,scale,budget):
    a,b,H=make_pair(image_source(pattern),angle=angle,scale=scale,blur=blur)
    def detect(im):
        p=cv2.goodFeaturesToTrack(im,budget,quality,distance,blockSize=5,useHarrisDetector=method=='Harris',k=.04)
        return np.empty((0,2)) if p is None else p[:,0,:]
    p,q=detect(a),detect(b);gt=project(p,H);valid=(gt>=0).all(1)&(gt<384).all(1)
    repeat=float((np.linalg.norm(gt[valid,None]-q[None],axis=2).min(1)<=3).mean()) if valid.any() and len(q) else 0.
    fig,axs=view([a,b],['Detected source points','Independently detected target points'])
    for ax,pts in zip(axs,[p,q]):
        if len(pts):ax.scatter(*pts.T,s=10,facecolors='none',edgecolors='red')
    return finish(fig,{'source':len(p),'target':len(q),'visible':int(valid.sum()),'repeatability@3px':repeat,'coverage':coverage(p)})

def show_patch(angle,scale,gain,bias,blur,noise,normalize,seed):
    a=facade(seed);patch=a[112:240,112:240];_,b,_=make_pair(patch,angle,scale,dx=0,dy=0,perspective=0,blur=blur,gain=1)
    rng=np.random.default_rng(seed);b=np.clip(b.astype(float)*gain+bias+rng.normal(0,noise,b.shape),0,255)
    u=patch[24:-24,24:-24].astype(float);v=b[24:-24,24:-24].copy()
    if normalize:u=(u-u.mean())/(u.std()+1e-9);v=(v-v.mean())/(v.std()+1e-9)
    fig,_=view([patch,b,np.abs(u-v)],['Source patch','Transformed patch','Compared central region: |difference|'])
    return finish(fig,{'RMS':float(np.sqrt(np.mean((u-v)**2))),'units':'standard deviations' if normalize else 'gray levels','pixels':int(u.size)})

def show_orientation(angle,scale,compensate,blur,noise,bins,cells,seed):
    # Known transform, oracle inverse. Teaches canonicalization, not an implementation of SIFT.
    a=facade(seed)[96:288,96:288];_,b,H=make_pair(a,angle,scale,dx=0,dy=0,perspective=0,blur=blur)
    b=np.uint8(np.clip(b.astype(float)+np.random.default_rng(seed).normal(0,noise,b.shape),0,255))
    aligned=cv2.warpPerspective(b,np.linalg.inv(H),(192,192),borderValue=128) if compensate else b
    def hist(im):
        im=im[48:144,48:144].astype(np.float32)/255
        gx=cv2.Sobel(im,cv2.CV_32F,1,0);gy=cv2.Sobel(im,cv2.CV_32F,0,1)
        mag,ori=cv2.cartToPolar(gx,gy);desc=[]
        for yy in np.array_split(np.arange(96),cells):
            for xx in np.array_split(np.arange(96),cells):
                ix=np.ix_(yy,xx);desc.extend(np.histogram(ori[ix],bins,range=(0,2*np.pi),weights=mag[ix])[0])
        d=np.array(desc);return d/(np.linalg.norm(d)+1e-9)
    d,e=hist(a),hist(aligned);fig,axs=plt.subplots(1,3,figsize=(12,3.5),constrained_layout=True)
    axs[0].imshow(b,cmap='gray');axs[0].set_title('Observed rotated/scaled patch');axs[1].imshow(aligned,cmap='gray');axs[1].set_title('Canonicalization using known H' if compensate else 'No canonicalization');axs[2].plot(d,label='source');axs[2].plot(e,label='target');axs[2].legend();axs[2].set_title('Cell orientation histograms')
    return finish(fig,{'descriptor L2':float(np.linalg.norm(d-e)),'dimension':len(d),'oracle inverse':compensate})

def show_metric(byte,c1,c2,c3,metric,flip,mask):
    candidates=np.array([c1,c2,c3],np.uint8);query=np.uint8(byte)^np.uint8(1<<flip if flip>=0 else 0)
    values=np.r_[query,candidates]&np.uint8(mask);bits=np.unpackbits(values).reshape(4,8)
    dist=np.sum(bits[1:]!=bits[0],axis=1).astype(float) if metric=='Hamming' else np.abs(values[1:].astype(float)-float(values[0]))
    fig,axs=plt.subplots(1,2,figsize=(12,3.6),constrained_layout=True)
    axs[0].imshow(bits,cmap='Greys',aspect='auto');axs[0].set(yticks=range(4),yticklabels=['query']+['candidate '+str(i+1) for i in range(3)],title='Masked bits (MSB first)')
    axs[1].bar(range(1,4),dist);axs[1].set(title=metric,ylabel='distance',xticks=[1,2,3])
    return finish(fig,{'query after flip':int(query),'distances':dist.tolist(),'nearest indices (ties included)':(np.flatnonzero(dist==dist.min())+1).tolist()})

def show_filter(d1,d2,competitor,ratio,mutual,duplicate,gt_error,gt_threshold):
    D=np.array([[d1,d2,12.],[competitor,10.,11.],[8.,9.,0.]])
    if duplicate:D[0,1]=d1
    order=np.argsort(D,axis=1,kind='stable');nn=order[:,0];first=D[np.arange(3),nn];second=D[np.arange(3),order[:,1]]
    ratio_ok=first<ratio*second;mutual_ok=np.argmin(D,axis=0)[nn]==np.arange(3);keep=ratio_ok & (mutual_ok if mutual else True)
    # GT diagnostic is intentionally independent of descriptor distances.
    errors=np.array([gt_error,8.,0.]);correct=errors<=gt_threshold
    fig,axs=plt.subplots(1,2,figsize=(12,3.5),constrained_layout=True);axs[0].imshow(D,cmap='viridis')
    for (i,j),v in np.ndenumerate(D):axs[0].text(j,i,f'{v:.1f}',ha='center',color='white')
    axs[0].set(title='Descriptor distances D',xlabel='target',ylabel='query')
    axs[1].bar(np.arange(3)-.18,first,width=.36,label='d1');axs[1].bar(np.arange(3)+.18,ratio*second,width=.36,label='ratio × d2');axs[1].legend();axs[1].set_title('Strict inequality; GT is a separate test')
    return finish(fig,{'nearest':nn.tolist(),'ratio pass':ratio_ok.tolist(),'mutual pass':mutual_ok.tolist(),'accepted':keep.tolist(),'GT correct':correct.tolist(),'precision':float(correct[keep].mean()) if keep.any() else 0.})

def show_matching(method,ratio,mutual,angle,blur,source,scale,budget,gain):
    a,b,H=make_pair(image_source(source),angle=angle,scale=scale,blur=blur,gain=gain)
    start=time.perf_counter();p,d1,metric=extract(a,method,budget);q,d2,_=extract(b,method,budget);detect_ms=(time.perf_counter()-start)*1000
    start=time.perf_counter();bf=cv2.BFMatcher(cv2.NORM_HAMMING if metric=='hamming' else cv2.NORM_L2)
    knn=bf.knnMatch(d1,d2,k=2) if len(d1) and len(d2)>=2 else []
    reverse={m.queryIdx:m.trainIdx for m in bf.match(d2,d1)} if len(d1) and len(d2) else {}
    pairs=np.array([(m.queryIdx,m.trainIdx) for item in knn if len(item)==2 for m,n in [item] if m.distance<ratio*n.distance and (not mutual or reverse[m.trainIdx]==m.queryIdx)],int).reshape(-1,2)
    match_ms=(time.perf_counter()-start)*1000;gt=project(p,H);g=gt[pairs[:,0]];err=np.linalg.norm(g-q[pairs[:,1]],axis=1);good=(err<=3)&(g>=0).all(1)&(g<384).all(1)
    report={'points':[len(p),len(q)],'matches':len(pairs),'good':int(good.sum()),'precision':float(good.mean()) if len(pairs) else 0.,'correct coverage':coverage(p[pairs[good,0]]),'detect+describe ms':detect_ms,'BF+filters ms':match_ms}
    draw_matches(a,b,p,q,pairs,good,title=f'{method}: {len(pairs)} matches, {good.sum()} correct');print(json.dumps(report));return report

def geometry_data(layout,noise,count,outliers,seed,perspective):
    rng=np.random.default_rng(seed);p=rng.uniform(25,359,(count,2))
    if layout=='clustered':p=160+(p-25)*64/334
    if layout=='line':p[:,1]=192
    H=np.array([[.95,-.12,28],[.1,.93,5],[perspective,-.00025,1.]])
    q=project(p,H)+rng.normal(0,noise,p.shape);n=int(count*outliers);bad=rng.permutation(count)[:n];q[bad]=rng.uniform(0,384,(n,2))
    return p,q,H,bad

def geometry_show(p,q,H,est,mask,title):
    yy,xx=np.mgrid[0:384:25j,0:384:25j];grid=np.c_[xx.ravel(),yy.ravel()]
    fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True);axs[0].scatter(*p.T,c=np.asarray(mask).ravel(),cmap='coolwarm',vmin=0,vmax=1);axs[0].set(xlim=(0,384),ylim=(384,0),aspect='equal',title=title)
    if est is None or not np.isfinite(est).all():
        axs[1].text(.5,.5,'Degenerate input: no homography',ha='center',va='center',transform=axs[1].transAxes);axs[1].axis('off')
        return finish(fig,{'status':'degenerate: no homography','points':len(p)})
    errors=np.linalg.norm(project(grid,est)-project(grid,H),axis=1);fit=np.linalg.norm(project(p,est)-q,axis=1)
    if not np.isfinite(errors).all():return finish(fig,{'status':'invalid projection'})
    im=axs[1].imshow(errors.reshape(25,25),extent=(0,384,384,0),vmin=0,vmax=max(2,float(np.percentile(errors,95))),cmap='magma');axs[1].set_title('Independent grid error (px)');fig.colorbar(im,ax=axs[1])
    return finish(fig,{'fit median px':float(np.median(fit)),'grid median px':float(np.median(errors)),'grid max px':float(errors.max()),'accepted':int(np.sum(mask))})

def show_spread(layout,noise,count,seed,perspective,solver,outliers,threshold):
    p,q,H,bad=geometry_data(layout,noise,count,outliers,seed,perspective)
    cv2.setRNGSeed(seed);est,mask=cv2.findHomography(p,q,cv2.RANSAC if solver=='RANSAC' else 0,threshold,maxIters=2000)
    if mask is None:mask=np.zeros(len(p))
    return geometry_show(p,q,H,est,mask,f'{layout}: {count} points, {len(bad)} outliers')

def show_ransac(outliers,noise,threshold,iterations,confidence,count,seed,layout):
    p,q,H,bad=geometry_data(layout,noise,count,outliers,seed,.0004)
    cv2.setRNGSeed(seed);est,mask=cv2.findHomography(p,q,cv2.RANSAC,threshold,maxIters=iterations,confidence=confidence)
    if mask is None:mask=np.zeros((len(p),1),np.uint8)
    accepted=mask.ravel().astype(bool);truth=np.ones(len(p),bool);truth[bad]=False
    report=geometry_show(p,q,H,est,mask,'RANSAC consensus (red) vs rejected (blue)')
    extra={'inlier precision':float(truth[accepted].mean()) if accepted.any() else 0.,'inlier recall':float(accepted[truth].mean()) if truth.any() else 0.,'actual clean fraction':float(truth.mean()),'theoretical draws for 4 clean':int(np.ceil(np.log(1-confidence)/np.log(1-truth.mean()**4))) if 0<truth.mean()<1 else 1}
    print(json.dumps(extra));return dict(report,**extra)

## Стенд 1. Отклик: плоская область, край и угол

Матрица M=Σ∇I∇Iᵀ описывает изменение яркости в окне. Два больших собственных значения означают наблюдаемость двух направлений. Harris: det(M)−k·tr(M)²; Shi–Tomasi: λmin. Числа вычислены по сумме градиентов, карты OpenCV нормированы иначе: абсолютные масштабы между ними не сравниваем.

In [ ]:
panel('response',show_response,{'pattern':W.Dropdown(description='pattern',**{'options': ['corner', 'edge', 'flat'], 'value': 'corner'}),'angle':W.IntSlider(description='angle',**{'value': 0, 'min': 0, 'max': 90, 'step': 15}),'contrast':W.FloatSlider(description='contrast',**{'value': 1, 'min': 0, 'max': 1, 'step': 0.1}),'noise':W.FloatSlider(description='noise',**{'value': 0, 'min': 0, 'max': 0.2, 'step': 0.02}),'window':W.IntSlider(description='window',**{'value': 9, 'min': 3, 'max': 25, 'step': 2}),'k':W.FloatSlider(description='k',**{'value': 0.04, 'min': 0.02, 'max': 0.15, 'step': 0.01}),'criterion':W.Dropdown(description='criterion',**{'options': ['Harris', 'Shi-Tomasi'], 'value': 'Harris'}),'seed':W.IntSlider(description='seed',**{'value': 4, 'min': 0, 'max': 20, 'step': 1})})

### Задания после стенда 1

1. При noise=0 сравните corner, edge и flat. Предскажите ранг и соотношение λmin/λmax; объясните связь с неопределённым направлением.
2. Снизьте contrast 1→.5, не меняя окно. Сопоставьте изменение собственных значений и Harris с их степенью по яркости.
3. Добавьте noise=.1 на flat. Почему положительный отклик ещё не означает устойчивую смысловую точку?
4. Для edge поверните angle 0→45 и затем измените окно. Проверьте устойчивость вывода о наблюдаемости. Сравните две карты критерия.

## Стенд 2. Детектор: порог, подавление соседей и повторяемость

quality — относительный порог, distance — минимальное расстояние между точками. Детекция независима на обоих кадрах. Направленная повторяемость — доля видимых проекций исходных точек, имеющих соседа во втором кадре в пределах 3 px; назначения 1:1 нет. Покрытие — занятые ячейки сетки 4×4. Это учебные диагностики, не официальный benchmark.

In [ ]:
panel('detector',show_detector,{'method':W.Dropdown(description='method',**{'options': ['Shi-Tomasi', 'Harris'], 'value': 'Shi-Tomasi'}),'quality':W.FloatSlider(description='quality',**{'value': 0.05, 'min': 0.01, 'max': 0.5, 'step': 0.01}),'distance':W.IntSlider(description='distance',**{'value': 8, 'min': 3, 'max': 30, 'step': 1}),'blur':W.FloatSlider(description='blur',**{'value': 0.5, 'min': 0, 'max': 3, 'step': 0.5}),'pattern':W.Dropdown(description='pattern',**{'options': ['textured', 'repeated', 'flat'], 'value': 'textured'}),'angle':W.IntSlider(description='angle',**{'value': 10, 'min': 0, 'max': 60, 'step': 5}),'scale':W.FloatSlider(description='scale',**{'value': 0.92, 'min': 0.6, 'max': 1.4, 'step': 0.04}),'budget':W.IntSlider(description='budget',**{'value': 600, 'min': 50, 'max': 1000, 'step': 50})})

### Задания после стенда 2

1. Предскажите эффект distance 3→20 на число точек и покрытие. Запишите оба показателя.
2. На том же входе измените quality .05→.3, затем отдельно budget 600→100. Разделите порог и ограничение числа.
3. При фиксированных настройках увеличьте blur .5→2.5, затем angle 10→50. Запишите видимый знаменатель и повторяемость.
4. Переключите Harris и Shi–Tomasi, затем flat без размытия. Объясните пустой результат; не объявляйте большее число точек лучшим качеством.

## Стенд 3. Сырой патч: фотометрия и геометрия

RMS сравнивает пиксели центральной области 80×80. Нормировка вычитает среднее и делит на стандартное отклонение отдельно в каждом патче. Она уменьшает влияние аффинного изменения яркости до клиппинга, но не компенсирует поворот. Границы исключены из сравнения; интерполяция и обрезка всё равно ограничивают инвариантность.

In [ ]:
panel('patch',show_patch,{'angle':W.IntSlider(description='angle',**{'value': 0, 'min': 0, 'max': 60, 'step': 5}),'scale':W.FloatSlider(description='scale',**{'value': 1, 'min': 0.7, 'max': 1.3, 'step': 0.1}),'gain':W.FloatSlider(description='gain',**{'value': 1, 'min': 0.5, 'max': 1.5, 'step': 0.1}),'bias':W.IntSlider(description='bias',**{'value': 0, 'min': -60, 'max': 60, 'step': 10}),'blur':W.FloatSlider(description='blur',**{'value': 0, 'min': 0, 'max': 3, 'step': 0.5}),'noise':W.IntSlider(description='noise',**{'value': 0, 'min': 0, 'max': 40, 'step': 5}),'normalize':W.Checkbox(description='normalize',**{'value': False}),'seed':W.IntSlider(description='seed',**{'value': 4, 'min': 0, 'max': 20, 'step': 1})})

### Задания после стенда 3

1. При тождественном преобразовании, нулевом шуме и blur=0 проверьте RMS=0.
2. Измените bias 0→20. Предскажите RMS без нормировки, затем включите её. Укажите единицы каждого результата.
3. Верните gain=1 и bias=0, поверните angle=30. Проверьте, компенсирует ли нормировка геометрическое изменение.
4. Отдельно измените blur и noise. Объясните различие изображений разности и почему нормировка не восстанавливает потерянные детали.

## Стенд 4. Канонизация и гистограммы градиентов

Учебный дескриптор разбивает центральную область на cells×cells и собирает bins направлений градиента, затем нормирует L2. Это не реализация SIFT. Компенсация использует известную H и обратный warp — идеализированный ориентир, а не найденную алгоритмом ориентацию. Ошибки интерполяции и утрата деталей остаются.

In [ ]:
panel('orientation',show_orientation,{'angle':W.IntSlider(description='angle',**{'value': 30, 'min': 0, 'max': 90, 'step': 5}),'scale':W.FloatSlider(description='scale',**{'value': 1, 'min': 0.7, 'max': 1.3, 'step': 0.1}),'compensate':W.Checkbox(description='compensate',**{'value': False}),'blur':W.FloatSlider(description='blur',**{'value': 0, 'min': 0, 'max': 3, 'step': 0.5}),'noise':W.IntSlider(description='noise',**{'value': 0, 'min': 0, 'max': 30, 'step': 5}),'bins':W.IntSlider(description='bins',**{'value': 8, 'min': 4, 'max': 16, 'step': 4}),'cells':W.IntSlider(description='cells',**{'value': 4, 'min': 1, 'max': 4, 'step': 1}),'seed':W.IntSlider(description='seed',**{'value': 4, 'min': 0, 'max': 20, 'step': 1})})

### Задания после стенда 4

1. При angle=0, scale=1, blur=noise=0 проверьте нулевую дистанцию. Вычислите размерность для cells=4, bins=8.
2. При angle=30 сравните compensate off/on. Объясните, какая информация использована для канонизации.
3. Измените cells 1→4 при одном преобразовании: какая пространственная информация теряется в общей гистограмме?
4. При включённой компенсации отдельно измените scale и blur. Почему известная H не гарантирует нулевой дистанции?

## Стенд 5. Бинарные тесты: Hamming и упакованные байты

Каждый бит — результат одного бинарного теста. Hamming считает разные биты; L2 над одним упакованным байтом равно модулю разности чисел и задаёт разный вес позициям. Маска AND исключает тесты. Показываются все равные ближайшие кандидаты. Это искусственный 8-битный пример, не ORB целиком.

In [ ]:
panel('metric',show_metric,{'byte':W.IntSlider(description='byte',**{'value': 0, 'min': 0, 'max': 255, 'step': 1}),'c1':W.IntSlider(description='c1',**{'value': 3, 'min': 0, 'max': 255, 'step': 1}),'c2':W.IntSlider(description='c2',**{'value': 128, 'min': 0, 'max': 255, 'step': 1}),'c3':W.IntSlider(description='c3',**{'value': 255, 'min': 0, 'max': 255, 'step': 1}),'metric':W.Dropdown(description='metric',**{'options': ['Hamming', 'byte L2'], 'value': 'Hamming'}),'flip':W.IntSlider(description='flip',**{'value': -1, 'min': -1, 'max': 7, 'step': 1}),'mask':W.IntSlider(description='mask',**{'value': 255, 'min': 0, 'max': 255, 'step': 1})})

### Задания после стенда 5

1. При query=0 и кандидатах 3/128/255 вычислите обе дистанции вручную, затем проверьте смену ближайшего соседа.
2. Верните Hamming и сравните flip=0 и flip=7 (−1 отключает flip). Почему вес бита в Hamming одинаков?
3. Измените mask 255→127: какой тест исключён и почему изменилось решение?
4. Сделайте c1=c2, затем mask=0. Запишите все ties и объясните, почему выбор одного argmin не снимает неоднозначность.

## Стенд 6. Ratio и mutual: разобрать каждое решение

Строки D — query, столбцы — target. Ratio использует два ближайших расстояния и строгое d1<ratio·d2. Mutual проверяет обратного ближайшего соседа. При ties индекс выбирается стабильно, но это не свидетельство достоверности. duplicate приравнивает второй элемент первой строки к d1. Параметры d1/d2 задают два элемента строки; при изменении их порядка ближайшие соседи пересортируются. GT-ошибка задана независимо и используется только для аудита.

In [ ]:
panel('filter',show_filter,{'d1':W.FloatSlider(description='d1',**{'value': 1, 'min': 0, 'max': 6, 'step': 0.5}),'d2':W.FloatSlider(description='d2',**{'value': 4, 'min': 0, 'max': 8, 'step': 0.5}),'competitor':W.FloatSlider(description='competitor',**{'value': 2, 'min': 0, 'max': 6, 'step': 0.5}),'ratio':W.FloatSlider(description='ratio',**{'value': 0.75, 'min': 0.3, 'max': 1, 'step': 0.05}),'mutual':W.Checkbox(description='mutual',**{'value': True}),'duplicate':W.Checkbox(description='duplicate',**{'value': False}),'gt_error':W.FloatSlider(description='gt_error',**{'value': 1, 'min': 0, 'max': 10, 'step': 0.5}),'gt_threshold':W.FloatSlider(description='gt_threshold',**{'value': 3, 'min': 1, 'max': 5, 'step': 0.5})})

### Задания после стенда 6

1. Для начальной D вручную найдите прямые/обратные NN, ratio pass, mutual pass и итог. Проверьте каждый массив.
2. Уменьшите competitor 2→.5: какую пару удалит mutual? Отключите mutual и объясните конкуренцию за target.
3. Поставьте d1=d2=0, затем d1=2,d2=4,ratio=.5. Проверьте двойной ноль и строгое неравенство.
4. Верните настройки, измените только gt_error 1→8. Какие решения фильтра останутся прежними? Почему precision изменится?

## Стенд 7. Реальные алгоритмы: общий вход и разные потери

ORB и AKAZE-MLDB используют Hamming, SIFT — L2. Извлечение независимо, бюджет ограничивает точки по отклику. Цвет — независимая известная H и ошибка ≤3 px, не решение фильтра. На рисунке до 75 линий; числа по всем парам. Время одного запуска разделено на два detect+describe и BF+filters, без подготовки данных и рисования; это не рейтинг скорости.

In [ ]:
panel('matching',show_matching,{'method':W.Dropdown(description='method',**{'options': ['ORB', 'SIFT', 'AKAZE'], 'value': 'ORB'}),'ratio':W.FloatSlider(description='ratio',**{'value': 0.75, 'min': 0.5, 'max': 1, 'step': 0.05}),'mutual':W.Checkbox(description='mutual',**{'value': True}),'angle':W.IntSlider(description='angle',**{'value': 12, 'min': 0, 'max': 60, 'step': 4}),'blur':W.FloatSlider(description='blur',**{'value': 0.5, 'min': 0, 'max': 3, 'step': 0.5}),'source':W.Dropdown(description='source',**{'options': ['textured', 'repeated', 'NASA photo', 'flat'], 'value': 'textured'}),'scale':W.FloatSlider(description='scale',**{'value': 0.9, 'min': 0.6, 'max': 1.4, 'step': 0.1}),'budget':W.IntSlider(description='budget',**{'value': 600, 'min': 100, 'max': 1000, 'step': 100}),'gain':W.FloatSlider(description='gain',**{'value': 1, 'min': 0.5, 'max': 1.5, 'step': 0.1})})

### Задания после стенда 7

1. На repeated сравните ratio .60→.95 при одном методе. Запишите matches/good/precision и покрытие правильными парами.
2. При зафиксированном ratio измените только mutual. Объясните изменение каждого показателя.
3. На NASA photo сравните все три метода с одинаковыми преобразованием и бюджетом. Приложите одинаковые кадры; бюджет не означает одинаковое число найденных точек.
4. Усильте angle, scale или blur по одному. Разделите потери обнаружения и сопоставления. Проверьте flat; он должен дать пустые пары, а не исключение.

## Стенд 8. Покрытие: подгонка и ошибка вне наблюдений

Известные правильные пары с шумом позволяют сравнить распределение точек. spread покрывает кадр, clustered занимает малую область, line вырожден. Независимая сетка 25×25 одинакова во всех состояниях. LS подгоняет все пары, RANSAC выделяет согласованный набор. Карта адаптирует шкалу до 95-го процентиля: сравнивайте числа и colorbar.

In [ ]:
panel('spread',show_spread,{'layout':W.Dropdown(description='layout',**{'options': ['spread', 'clustered', 'line'], 'value': 'spread'}),'noise':W.FloatSlider(description='noise',**{'value': 1, 'min': 0, 'max': 3, 'step': 0.5}),'count':W.IntSlider(description='count',**{'value': 20, 'min': 4, 'max': 100, 'step': 4}),'seed':W.IntSlider(description='seed',**{'value': 8, 'min': 0, 'max': 20, 'step': 1}),'perspective':W.FloatSlider(description='perspective',**{'value': 0.0004, 'min': 0, 'max': 0.001, 'step': 0.0001}),'solver':W.Dropdown(description='solver',**{'options': ['LS', 'RANSAC'], 'value': 'LS'}),'outliers':W.FloatSlider(description='outliers',**{'value': 0, 'min': 0, 'max': 0.5, 'step': 0.1}),'threshold':W.FloatSlider(description='threshold',**{'value': 3, 'min': 0.5, 'max': 8, 'step': 0.5})})

### Задания после стенда 8

1. При noise=0,outliers=0 сравните spread/clustered, затем noise=1. Запишите fit и grid error.
2. Для clustered увеличьте count. Всегда ли больше локальных наблюдений заменяет покрытие кадра? Проверьте на двух seed.
3. При spread добавьте outliers=.2 и сравните LS/RANSAC. Разделите распределение и устойчивость к выбросам.
4. Выберите line. Объясните сообщение о вырождении и почему увеличение count вдоль прямой не определяет гомографию плоскости.

## Стенд 9. RANSAC: согласие модели и независимая проверка

Четыре пары — минимальная невырожденная выборка для H. Для чистой доли w вероятность чистой выборки равна w⁴; оценка числа попыток log(1−confidence)/log(1−w⁴) предполагает независимость и невырожденность. maxIters — предел; алгоритм может закончить раньше. Порог RANSAC меняет внутренний consensus, но GT и сетка оценки фиксированы.

In [ ]:
panel('ransac',show_ransac,{'outliers':W.FloatSlider(description='outliers',**{'value': 0.3, 'min': 0, 'max': 0.8, 'step': 0.1}),'noise':W.FloatSlider(description='noise',**{'value': 1, 'min': 0, 'max': 4, 'step': 0.5}),'threshold':W.FloatSlider(description='threshold',**{'value': 3, 'min': 0.5, 'max': 12, 'step': 0.5}),'iterations':W.IntSlider(description='iterations',**{'value': 500, 'min': 10, 'max': 3010, 'step': 100}),'confidence':W.FloatSlider(description='confidence',**{'value': 0.99, 'min': 0.8, 'max': 0.999, 'step': 0.001}),'count':W.IntSlider(description='count',**{'value': 60, 'min': 12, 'max': 120, 'step': 12}),'seed':W.IntSlider(description='seed',**{'value': 8, 'min': 0, 'max': 20, 'step': 1}),'layout':W.Dropdown(description='layout',**{'options': ['spread', 'clustered', 'line'], 'value': 'spread'})})

### Задания после стенда 9

1. При фиксированных входных данных сравните threshold .5→3→12. Запишите accepted, inlier precision/recall и grid error.
2. При outliers=.7 сравните iterations=10 и 2010 на трёх seed. Сопоставьте с теоретической оценкой; не считайте один seed доказательством.
3. При малом пороге увеличьте noise 0→3: почему верные по происхождению пары могут выйти из consensus?
4. Сравните spread/clustered/line. Почему большое число inliers не гарантирует точного преобразования всей плоскости? Сформулируйте, что нужно проверить перед геометрией камеры.

## Переход к самостоятельному эксперименту
В `02_matching_experiment.ipynb` реализуйте пять функций и выполните val → фиксацию → test. К отчёту приложите таблицу 36 пунктов демо. Настройки демо не заменяют выбор по val. Гомография здесь связывает плоские изображения; сцены с параллаксом требуют следующей лабораторной.

## Источники и границы вывода

- [OpenCV: feature matching](https://docs.opencv.org/4.13.0/dc/dc3/tutorial_py_matcher.html): L2 для SIFT, Hamming для ORB (`WTA_K=2`) и используемого AKAZE-MLDB. Детектор выбирает координаты, дескриптор описывает окрестность; это разные этапы.
- [OpenCV: Harris](https://docs.opencv.org/4.13.0/dc/d0d/tutorial_py_features_harris.html) и [гомография/RANSAC](https://docs.opencv.org/4.13.0/d1/de0/tutorial_py_feature_homography.html): отклик и геометрическая проверка.
- [Lowe, 2004](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf): SIFT и отношение расстояний до двух ближайших соседей. Число .75 здесь — вариант для опыта, не универсальный порог.
- [OpenCV: AKAZE](https://docs.opencv.org/4.13.0/db/d70/tutorial_akaze_matching.html): используем стандартный бинарный MLDB.
- [NASA / scikit-image astronaut](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.astronaut): открытое фото Eileen Collins, public domain; закрепленный файл scikit-image v0.25.2 проверяется SHA-256. Фото скачивается во временный кэш, в notebook не встроено. Объект интереса — локальные текстуры, личность не распознается.

Синтетический фасад нарисован учебным кодом; второй кадр фотографии вычислен известной гомографией. Это контролируемый опыт на плоском изображении, а не две реальные фотографии 3D-сцены и не результат HPatches. Он не проверяет параллакс, изменение видимости или независимость новых сцен.